<span style="color:red; font-size:40px"> Conditioning of Linear Algebra Problems </span>

<font size = "6">

**Square Linear Systems**

<font size = "4">

- We deviate from the "$x = \textrm{input}$" and "$y = \textrm{output}$" paradigm, and follow standard notational conventions from linear algebra.

- Problem: solve $\mathbf{A}\mathbf{x} = \mathbf{b}$ for a nonsingular $n\times n$ matrix $\mathbf{A}$.

- If we are analyzing the effects of perturbations in the right-hand side, then our goal is to compute $\mathbf{x} = f(\mathbf{b}) = \mathbf{A}^{-1}\mathbf{b}$, and if we use a backward stable algorithm, we actually compute $\hat{\mathbf{x}} = f(\hat{\mathbf{b}}) = \mathbf{A}^{-1}\hat{\mathbf{b}}$.

- If we are analyzing the effects of perturbations in the *matrix*, then our goal is to compute $\mathbf{x} = f(\mathbf{A}) = \mathbf{A}^{-1}\mathbf{b}$, and if we use a backward stable algorithm, we actually compute $\hat{\mathbf{x}} = f(\hat{\mathbf{A}}) = \hat{\mathbf{A}}^{-1}\mathbf{b}$.

- Or we could consider both: $\mathbf{x} = f(\mathbf{A}, \mathbf{b}) = \mathbf{A}^{-1}\mathbf{b}$

- In any case, the condition number for this problem is:

$$\textrm{cond}(f) = \textrm{cond}(\mathbf{A}) = \|\mathbf{A}\|\cdot\|\mathbf{A}^{-1}\|$$

- Above, $\|\cdot\|$ is an unspecified induced matrix norm. The most commonly used are:

    - $\textrm{cond}_1(\mathbf{A}) = \|\mathbf{A}\|_1\cdot\|\mathbf{A}^{-1}\|_1$

    - $\textrm{cond}_2(\mathbf{A}) = \|\mathbf{A}\|_2\cdot\|\mathbf{A}^{-1}\|_2$

    - $\textrm{cond}_\infty(\mathbf{A}) = \|\mathbf{A}\|_\infty\cdot\|\mathbf{A}^{-1}\|_\infty$

In [ ]:
import numpy as np
import numpy.linalg as la 

A = np.array([[2, -1, 1], [1, 0, 1], [3, -1, 4]])

print("1-norm condition #:", la.cond(A,1))
print("2-norm condition #:", la.cond(A,2))
print("2-norm condition #:", la.cond(A))
print("inf-norm condition #:", la.cond(A,np.inf))


<font size = "4">

If $\mathbf{A}$ has no inverse i.e., it is singular, then $\textrm{cond}(\mathbf{A}) = \infty$ by definition.

In [ ]:
A = np.array([[0, 0], [1, 1]])
display(A)
print()
print(la.cond(A))

<font size = "4">

If numpy returns a huge condition number, it means the matrix is most likely singular.

In [ ]:
# this matrix is exactly singular
A = np.array([[1, 1], [2, 2]])

display(A)
print()
print(la.cond(A))

<font size = "4">

- This is the condition number of solving a square linear system, whether considering errors in the matrix or the right-hand side:

$$\frac{\|\hat{\mathbf{x}} - \mathbf{x}\|}{\|\mathbf{x}\|} \leq \textrm{cond}(\mathbf{A})\frac{\|\hat{\mathbf{b}} - \mathbf{b}\|}{\|\mathbf{b}\|}$$

$$\frac{\|\hat{\mathbf{x}} - \mathbf{x}\|}{\|\mathbf{x}\|} \leq \textrm{cond}(\mathbf{A})\frac{\|{\mathbf{\hat{A}}} - \mathbf{A}\|}{\|\mathbf{A}\|}$$

- If we consider a linear system where the input data is accurate up to the level of $\epsilon_{\textrm{machine}}$, then the approximate solution will satisfy 

$$\frac{\|\hat{\mathbf{x}} - \mathbf{x}\|}{\|\mathbf{x}\|} \lesssim \textrm{cond}(\mathbf{A})\cdot \epsilon_{\textrm{machine}} \approx 10^{-16}\cdot\textrm{cond}(\mathbf{A})$$

<font size = "4">

**Constructing a well-conditioned matrix**

- Recall that $\|\mathbf{A}\|_2 = \sigma_{\textrm{max}}$ = largest singular value of $\mathbf{A}$

- Also, $\|\mathbf{A}^{-1}\|_2 = \dfrac{1}{\sigma_{\textrm{min}}}$ = reciprocal of smallest singular value of $\mathbf{A}$

- So $\textrm{cond}_2(\mathbf{A}) = \dfrac{\sigma_{\textrm{max}}}{\sigma_{\textrm{min}}}$

- So we get a well-conditioned matrix if we can guarantee that this ratio is manageable.

In [ ]:
# Generate well-conditioned matrix
n = 25

A_1 = np.random.rand(n,n)
A_2 = np.random.rand(n,n)

# we just need the Q_i's to get orthgonal matrices
Q_1, R_1 = la.qr(A_1)
Q_2, R_2 = la.qr(A_2)

# lower and upper bounds on range of singular values
sig_lb = 1
sig_ub = 20

sig = np.random.uniform(sig_lb, sig_ub, n)

A = Q_1 @ np.diag(sig) @ Q_2.T

print("Condition #:", la.cond(A)) # 2-norm condition number
print("Ratio of singular values:", sig.max()/sig.min())

In [ ]:
x_true = np.random.randn(n,)

b = A @ x_true

x_computed = la.solve(A, b)

print("relative error:", la.norm(x_computed - x_true)/la.norm(x_true))
print("cond(A) x eps:", la.cond(A) * np.finfo(np.float64).eps)

<font size = "4">

**Generating well-conditioned "pivot-free" matrices**

- Generate a well-conditioned matrix that won't cause this algorithm to fail:

<div style="text-align: center;">
  <img src="files/gauss_algorithm.png" alt="Centered image" width = "550">
  <br>
  <br>
  <figcaption>
  <font size = "2"> Michael T. Heath (Scientific Computing: An Introductory Survey)</figcaption>
</div>

<br>

In [ ]:
def lu_factor_in_place(A):
    n = A.shape[0]
    for k in range(n):
        for i in range(k+1, n):
            A[i,k] = A[i,k] / A[k,k]
        
        for j in range(k+1, n):
            for i in range(k+1, n):
                A[i,j] = A[i,j] - A[i,k]*A[k,j]


In [ ]:
# Generate well-conditioned matrix
n = 25

A_1 = np.random.rand(n,n)
A_2 = np.random.rand(n,n)

# we just need the Q_i's to get orthgonal matrices
Q_1, R_1 = la.qr(A_1)
Q_2, R_2 = la.qr(A_2)

# lower and upper bounds on range of singular values
sig_lb = 1
sig_ub = 20

sig = np.random.uniform(sig_lb, sig_ub, n)

A = Q_1 @ np.diag(sig) @ Q_2.T


import scipy.linalg as sla 

# compute A = PLU factorization
P, L, U = sla.lu(A)

# P.T @ A = LU (no pivoting)
A = P.T @ A


A_copy = A.copy()
lu_factor_in_place(A_copy)


# extract LU factors
L = np.eye(n) + np.tril(A_copy, -1)
U = np.triu(A_copy)

print(la.norm(L@U - A)/la.norm(A))



<font size = "6">

**Rectangular Least-Squares Problems**

<font size = "4">

For $\mathbf{A} \in \mathbb{R}^{m\times n}$ with $m > n$, and $\mathbf{b} \in \mathbb{R}^m$, we want to solve the problem:

$$\argmin_{\mathbf{x}\in \mathbb{R}^n} \|\mathbf{b} - \mathbf{A}\mathbf{x}\|_2$$

- This *can* be solved by transforming it to the problem: solve $\mathbf{A}^T\mathbf{A}\mathbf{x} = \mathbf{A}^T\mathbf{b}$.

- This has the solution $\mathbf{x} = \left(\mathbf{A}^T\mathbf{A}\right)^{-1}\mathbf{A}^T\mathbf{b} = \mathbf{A}^+\mathbf{b}$.

- $\mathbf{A}^+ = \left(\mathbf{A}^T\mathbf{A}\right)^{-1}\mathbf{A}^T$ is the *pseudo-inverse* of $\mathbf{A}$

- For a rectangular matrix $\mathbf{A}$, define the condition number as

$$\textrm{cond}(\mathbf{A}) = \textrm{cond}(\mathbf{A}^+)$$


**Condition Number of a Least-Squares Problem**

- In addition to $\textrm{cond}(\mathbf{A})$, the condition number of a least-squares **problem** also depends on the angle $\theta$ between the vector $\mathbf{b}$ and the column space of $\mathbf{A}$.

- The condition number with respect to changes in $\mathbf{b}$ is:

$$\frac{\textrm{cond}(\mathbf{A})}{\cos(\theta)}$$

- The condition number with respect to changes in $\mathbf{A}$ is:

$$\textrm{cond}(\mathbf{A}) + \left[\textrm{cond}(\mathbf{A})\right]^2\cdot \tan(\theta)$$

- In either case, if $\theta \approx 0$, then the condition number (approximately) reduces to $\textrm{cond}(\mathbf{A})$.

**Condition Number of the Normal Equations**

- Transforming the problem to $\mathbf{A}^T\mathbf{A}\mathbf{x} = \mathbf{A}^T\mathbf{b}$ results in the condition number being changed to that of a square linear system. So the condition number is

$$\textrm{cond}(\mathbf{A}^T\mathbf{A}) = \left[\textrm{cond}(\mathbf{A})\right]^2$$

- This can lead to a much more ill-conditioned problem.



In [ ]:
m = 100
n = 15

t = np.linspace(0, 1, m)

A = np.zeros((m,n))
for j in range(n):
    A[:,j] = t**j

print(la.cond(A,2))
print(la.cond(A.T @ A,2))

In [ ]:
b = np.exp(np.sin(4*t))

b = b/2006.787453080206; # chosen so x[14] = 1

# "exact" solution
x_lstsq, res, rank, s_vals = la.lstsq(A, b)


# solution using normal equations and np.linalg.solve
ATA = A.T @ A
c = A.T @ b
x_normal = la.solve(ATA, c) 


# solution using QR factorization and np.linalg.solve
Q, R = la.qr(A)
x_qr = la.solve(R, Q.T @ b)


In [ ]:
cos_theta = la.norm(A @ x_lstsq)/la.norm(b)
print(cos_theta)
tan_theta = la.norm(b - A @ x_lstsq)/la.norm(A @ x_lstsq)
print(tan_theta)

In [ ]:
eps = np.finfo(np.float64).eps
k_A = la.cond(A,2)

print(la.norm(x_qr - x_lstsq)/la.norm(x_lstsq))
print(k_A/cos_theta * eps)
print((k_A + tan_theta*k_A**2) * eps)

In [ ]:
eps = np.finfo(np.float64).eps
k_ATA = la.cond(A.T @ A,2)

print(la.norm(x_normal - x_lstsq)/la.norm(x_lstsq))
print(k_ATA * eps)